<h1>Sarcasm detection system using custom
Naïve Bayes implementations</h1>

In [4]:
import numpy as np
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt
import re
import json
import math 
from collections import defaultdict, Counter
from sklearn.metrics  import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [6]:
student_ID = 276690
last_6     = student_ID % 1000000
seed       = last_6 % 100000

In [9]:
df = pd.read_json('/Users/arjun/Downloads/Data/Sarcasm_Headlines_Dataset_v2.json', lines=True)
print(df.shape)
df.head()



(28619, 3)


,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [12]:
#shuffling
df_shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
df_shuffled.head()

,is_sarcastic,headline,article_link
0,1,woman walking alone at night picks up pace aft...,https://local.theonion.com/woman-walking-alone...
1,0,exxon mobil told to hand over decades of clima...,https://www.huffingtonpost.com/entry/exxon-mas...
2,0,top german soccer team hit in explosions,https://www.huffingtonpost.com/entry/borussia-...
3,0,assad's forces take 'capital of revolution',https://www.huffingtonpost.com/entry/assad-tak...
4,1,you to receive 15 pounds of venison sausage fr...,https://www.theonion.com/you-to-receive-15-pou...


In [13]:
#70/15/15 split 
n = len(df_shuffled)
training_upper_lim = int(0.7 * n)
validation_upper_lim = int(0.85 * n)
training_dataset = df_shuffled.iloc[: training_upper_lim]
validation_dataset = df_shuffled.iloc[training_upper_lim:validation_upper_lim]
test_dataset = df_shuffled.iloc[validation_upper_lim:]

In [14]:
print(n)
print(f'train dataset count : {len(training_dataset)}')
print(f'validation dataset count : {len(validation_dataset)}')
print(f'test dataset count : {len(test_dataset)}')
total = len(training_dataset) + len(validation_dataset) + len(test_dataset)
print(f'{'count verified'if total ==n else 'mismatch in count'}')




28619
train dataset count : 20033
validation dataset count : 4293
test dataset count : 4293
count verified


In [15]:
#smoothing
smoothing_sets = {
    0: [0, 0.1, 1],
    1: [0, 0.5, 2],
    2: [0, 1,   5],
}
k = seed % 4 
smoothing_key    = seed % 3
selected_alphas  = smoothing_sets[smoothing_key]

**Task 1 – Tokenisation & Preprocessing**

In [16]:
#basic cleaning
def basic_cleaning (text):
     text = text.lower() #remove stand alone number
     text = re.sub(r"<.*?>","", text) #html
     text = re.sub(r'[?!]*\?[!]+[?!]*|[?!]*![?]+[?!]*', '[INT_MARK]', text)
     text = re.sub(r'!{2,}','[EXC_SEQ]',  text) #Exclamation Sequences: !!, !!!, etc. à[EXC_SEQ]
     text = re.sub(r'\?{2,}','[QUE_SEQ]',  text)  #Question Sequences: ??, ???, etc. à[QUE_SEQ]
     text = re.sub(r'\.{3,}',   '[ELLIP]',    text) #Ellipses: ... à [ELLIP]

     text = re.sub(r"[^\w\s'\[\]]", "", text)  # remove non word/space/apostrophe
     text = re.sub(r"(?<!\w)'|'(?!\w)", '', text)  # remove apostrophes NOT inside words

     text = re.findall(r'\[.*?\]|\w+(?:\'\w+)*', text)
     return text
     

references:
https://www.geeksforgeeks.org/python/how-to-remove-html-tags-from-string-in-python/

https://regex101.com/

https://docs.python.org/3/library/re.html#re.sub #pattern substitution


In [20]:
df_shuffled['headline'].iloc[1]

'exxon mobil told to hand over decades of climate documents in major legal blow'

In [21]:
#testing regex
sample_headlines = [
    "Scientists DISCOVER!!! That Water Is WET!!",   
    "Are? you?? serious????",             
    "Oh reaLly?! come on!!??",      
    "just waiting...",    
    "<b>Breaking news</b>: bomb blast", 
    "senate's new plan to [repeal] but not replace",  
    "a 'damn' letter from a serial killer",    
]

data=[]
for i in sample_headlines:
    word = basic_cleaning(i)
    print(word)




['scientists', 'discover', '[EXC_SEQ]', 'that', 'water', 'is', 'wet', '[EXC_SEQ]']
['are', 'you', '[QUE_SEQ]', 'serious', '[QUE_SEQ]']
['oh', 'really', '[INT_MARK]', 'come', 'on', '[INT_MARK]']
['just', 'waiting', '[ELLIP]']
['breaking', 'news', 'bomb', 'blast']
["senate's", 'new', 'plan', 'to', '[repeal]', 'but', 'not', 'replace']
['a', 'damn', 'letter', 'from', 'a', 'serial', 'killer']


**Step 2 – Negation Handling**

In [22]:
#Negation Handling
negation_words={ "not", "no", "never", "don't", "didn't", "isn't", "wasn't"}
stop_words={"the", "a","also","an","and","are","as","at","be","because","been","but",
            "by","for","from","has","have","however","if","in","is","of",
            "on","or","so","than","that","their","there","these","this","to","was","were"}
vowels = set('aeiou')

def count_vowels(token):
    return sum(1 for ch in token if ch in vowels )
    
def negation_handling(tokens):
    result = []
    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token in negation_words:
            result.append(token)
            n_vowels = count_vowels(token)
            i += 1
            for _ in range(n_vowels):
                if i < len(tokens):
                    result.append("NOT_" + tokens[i])
                    i += 1
        else:
            result.append(token)
            i += 1
    return result
    


In [23]:

def stopword_removal(tokens):
    return [t for t in tokens if t.startswith("NOT_") or t not in stop_words]
 
 
def generate_ngrams(tokens):
    unigrams = tokens
    bigrams  = [f"{tokens[i]}__{tokens[i+1]}" for i in range(len(tokens) - 1)] #double underscore
    return unigrams + bigrams
 

In [24]:
def replace_oov(tokens, vocab):
    return [t if t in vocab else "<UNK>" for t in tokens]

In [25]:
def preprocess(text, vocab=None, ablation_k=None):
    tokens = basic_cleaning(text)

    if ablation_k != 2:
        tokens = negation_handling(tokens)

    if ablation_k != 3:
        tokens = stopword_removal(tokens)

    all_tokens = generate_ngrams(tokens)

    if ablation_k == 0:
        all_tokens = [t for t in all_tokens if "_" in t]
    elif ablation_k == 1:
        all_tokens = [t for t in all_tokens if "_" not in t]

    if vocab is not None:
        all_tokens = replace_oov(all_tokens, vocab)

    return all_tokens

**Step 5 – OOV Handling**

In [26]:
#OOV Handling 

train_tokens = training_dataset["headline"].apply(lambda x: preprocess(x))
vocab            = set(tok for doc in train_tokens for tok in doc)
train_tokens = training_dataset["headline"].apply(lambda x: preprocess(x, vocab))
val_tokens   = validation_dataset["headline"].apply(lambda x: preprocess(x, vocab))
test_tokens  = test_dataset["headline"].apply(lambda x: preprocess(x, vocab))

In [27]:


val_tokens.head()


20033    [two, nonbinary, college, activists, creating,...
20034    [jill, <UNK>, saw, <UNK>, singer's, divorce, c...
20035    [study, major, shift, media, landscape, occurs...
20036    [new, healthier, menu, features, food, wendy's...
20037    [bob, dole, picked, off, large, hawk, circling...
Name: headline, dtype: object

In [28]:
test_tokens.head()

24326    [wildfires, force, colorado, <UNK>, rocky, mou...
24327    [best, chance, defeat, roy, moore, may, democr...
24328                          [pentagon, planning, <UNK>]
24329    [no, NOT_more, adult, conversations, prayers, ...
24330    [alabama, native, channing, tatum, encourages,...
Name: headline, dtype: object

In [29]:
train_tokens.head()

0    [woman, walking, alone, night, picks, up, pace...
1    [exxon, mobil, told, hand, over, decades, clim...
2    [top, german, soccer, team, hit, explosions, t...
3    [assad's, forces, take, capital, revolution, a...
4    [you, receive, 15, pounds, venison, sausage, u...
Name: headline, dtype: object

<h1>Task 2 – Manual Feature Representation</h1>

In [30]:
#Compute Document Frequency
def compute_doc_freq(tokenised_doc):
    doc_freq_count ={}
    for doc in tokenised_doc:
        for tkn in set(doc): #to remove th duplicates
            doc_freq_count[tkn]= doc_freq_count.get(tkn,0) + 1

    return doc_freq_count
doc_freq_count = compute_doc_freq(train_tokens.tolist())

#print(doc_freq_count)

In [31]:
#Rare Word Filtering
#Count Vectoriser
t = (seed % 6) + 2 #this clean up the noise, any word appearing less than t value will be removed

print(f"seed: {seed}")
print(f"seed mod 6: {seed % 6}")
print(f"threshold: {t}")

seed: 76690
seed mod 6: 4
threshold: 6


In [32]:
#sorting in alphabetical order
filtered_tokens = [token for token, count in doc_freq_count.items() if count >= t]
filtered_tokens = sorted(filtered_tokens)
#print(filtered_tokens)

In [33]:
#prime position indexing
def find_prime_number(n):
    prime_list =[True]* (n+1)
    prime_list[0]=False
    prime_list[1]=False
    for i in range(2, int(n**0.5)+1):
        if prime_list[i]:
            for j in range(i*i,n+1,i):
                prime_list[j]=False
    return {i for i in range(2,n+1) if prime_list[i]}
primes_list = find_prime_number(len(filtered_tokens) +10) #buffer added just in case

feature_map = {0: "NON_PRIME"} # colomn 0 is always not prime
for position, token in enumerate(filtered_tokens,start=1):
    if position in primes_list:
        feature_map[position] =token #seperate  colomns for prime number
colomn_order = sorted(feature_map.keys())
prime_positions = set(feature_map.keys()) - {0} #since 0 is already non prime
token_matrix = {tkn: pos for pos,tkn in enumerate(filtered_tokens,start=1)}

In [34]:
print(f"Column 0: {feature_map[0]}")
print(f"Column 2: {feature_map.get(2)}")
print(f"col 0 in prime_positions: {0 in prime_positions}")
print(f"col 2 in prime_positions: {2 in prime_positions}") 

Column 0: NON_PRIME
Column 2: 10
col 0 in prime_positions: False
col 2 in prime_positions: True


In [36]:


#needed some changes in the above code , Vocab size is only 5299/28000 headlines which is very low




In [37]:
def build_count_matrix(tokenised_docs):
    rows=[]
    for doc in tokenised_docs:
        counts = {}
        for token in doc:
            pos=token_matrix.get(token)
            if pos is None:
                continue
            if pos in prime_positions:
                counts[pos]=counts.get(pos,0) +1
            else:
                counts[0] = counts.get(0,0) +1
        rows.append([counts.get(c,0) for c in colomn_order])
    return np.array(rows)


In [38]:
train_data_matrix = build_count_matrix(train_tokens.tolist())
test_data_matrix = build_count_matrix(test_tokens.tolist())
validation_data_matrix = build_count_matrix(val_tokens.tolist())


In [39]:
print(f"train_count: {train_data_matrix.shape}")
print(f"val_count: {test_data_matrix.shape}")
print(f"test_count: {validation_data_matrix.shape}")

train_count: (20033, 771)
val_count: (4293, 771)
test_count: (4293, 771)


**B. TF-IDF**

In [40]:
def compute_tf(doc):
    tf ={}
    total_words = len(doc) if len(doc) > 0 else 1
    for tkn in doc :
        tf[tkn]= tf.get(tkn,0) +1
    for tkn in tf:
        tf[tkn] = tf[tkn]/total_words
    return tf

def compute_idf(df_count, n):
    idf ={}
    for tkn,doc_freq in df_count.items():
        idf[tkn]=math.log((n+1)/(doc_freq+1)) +1
    return idf

num_train_docs = len(training_dataset)
idf = compute_idf(doc_freq_count, num_train_docs)
sample_words = list(idf.items())[:5]
for word, score in sample_words:
    print(f"  '{word}' => IDF = {score:.4f}")

  'night__picks' => IDF = 10.2120
  'alabama' => IDF = 8.0148
  'picks' => IDF = 7.9095
  'spotting' => IDF = 10.2120
  'picks__up' => IDF = 8.7080


In [41]:
def compute_tf_idf(tokenised_docs, idf, filtered_tokens):
    tkn_index = {t: i for i, t in enumerate(filtered_tokens)}
    rows = []
    for doc in tokenised_docs:
        vct = np.zeros(len(filtered_tokens))
        tf  = compute_tf(doc)
        for tkn, tf_score in tf.items():
            if tkn in tkn_index:     
                index      = tkn_index[tkn]
                idf_score  = idf.get(tkn, 0)
                vct[index] = tf_score * idf_score
        rows.append(vct)
    return np.array(rows)

# TF-IDF
train_tf_idf = compute_tf_idf(train_tokens.tolist(), idf, doc_freq_count)
val_tf_idf   = compute_tf_idf(val_tokens.tolist(),   idf, doc_freq_count)
test_tf_idf  = compute_tf_idf(test_tokens.tolist(),  idf, doc_freq_count)

print(f"{train_tf_idf.shape}")
print(f"tfidf matrix : {train_tf_idf[0][:10].round(4)}") #first 


(20033, 145956)
tfidf matrix : [0.3294 0.2585 0.2551 0.3294 0.2809 0.2536 0.1667 0.2568 0.3294 0.3294]


**C. Using the training dataset, report the top 10 tokens with the highest Document**

In [34]:
top10 = sorted(doc_freq_count.items(), key=lambda x: -x[1])[:10]
top10_df = pd.DataFrame(top10, columns=['Token', 'Document Frequency'])
print(top10_df.to_string(index=False))


Token  Document Frequency
 with                1366
  new                1143
trump                 996
  man                 941
about                 766
  you                 699
after                 694
  out                 606
   up                 598
   it                 586


<h1> Task 3 — Naive Bayes Variants</h1>